# Fidelity Validation in Gendantic

Gendantic samples distribution-annotated fields to match their spec. **Fidelity validation lets you *prove* that promise** rather than trusting it.

`fidelity_report(records, model_class)` compares a batch of generated records against the model's declared distributions and correlations and returns a structured, printable report. It **never raises** — you inspect `.passed` (or the per-field / per-correlation results) and decide what to do.

It's a great fit for tests and CI: it catches regressions in the sampling machinery, and because sampling is deterministic from a seed, the checks are reproducible.

In [ ]:
from typing import Annotated

import numpy as np
import matplotlib.pyplot as plt
from pydantic import BaseModel

from gendantic import (
    DistributionSampler,
    LLMDrivenModelAnalyser,
    fidelity_report,
    Normal,
    Uniform,
    Categorical,
    Poisson,
    Beta,
    Correlations,
)

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)

## Generating data to check

In a real pipeline you'd validate the output of `generate_synthetic_data(...)`. Here we sample directly with `DistributionSampler` so the notebook is **self-contained and LLM-free** — fidelity checks work identically on either source, since both produce records shaped by the same distribution specs.

In [ ]:
def sample_records(model, count, seed=42):
    """Generate spec-compliant records without an LLM.

    In a real pipeline you would pass the output of ``generate_synthetic_data``
    here instead; ``fidelity_report`` treats model instances and dicts alike.
    """
    specs = LLMDrivenModelAnalyser.extract_distribution_specs(model)
    correlations = getattr(model, "__correlations__", None)
    return DistributionSampler(seed=seed).sample_fields(specs, count, correlations)

## A model with a mix of distributions

This `Employee` exercises every kind of check: continuous (`Normal`, `Uniform`, `Beta`), a discrete count (`Poisson`), a categorical, and a declared correlation between `age` and `salary`.

In [ ]:
class Employee(BaseModel):
    salary: Annotated[float, Normal(mean=75000, std=20000)]
    age: Annotated[float, Uniform(min=22, max=65)]
    department: Annotated[
        str, Categorical(weights={"Engineering": 0.5, "Sales": 0.3, "Support": 0.2})
    ]
    tickets_closed: Annotated[int, Poisson(lam=8.0)]
    performance: Annotated[float, Beta(alpha=5, beta=2)]

    __correlations__ = Correlations(
        ("age", "salary", 0.6),
    )


records = sample_records(Employee, count=2000)
records[0]

## The report

Printing the report gives a human-readable table. Each field is checked with the test appropriate to its distribution:

| Kind | Distributions | Test |
|------|---------------|------|
| Continuous | normal, uniform, lognormal, exponential, beta | Kolmogorov-Smirnov vs. the theoretical CDF |
| Discrete counts | poisson, binomial | Chi-square goodness-of-fit on the count histogram |
| Categorical | categorical | Chi-square on observed vs. expected frequencies |

Correlations are checked with **Spearman** (the rank correlation copulas actually control), with Pearson reported alongside.

In [ ]:
report = fidelity_report(records, Employee)
print(report)

## Reading the results programmatically

`report.passed` is the overall verdict. `report.fields` and `report.correlations` hold the per-item detail, so you can assert on exactly what you care about.

In [ ]:
print(f"Overall passed: {report.passed}\n")

for f in report.fields:
    status = "PASS" if f.passed else "FAIL"
    print(f"{f.field:15s} {f.distribution:12s} {f.test:5s} p={f.p_value:6.3f}  [{status}]")

print()
for c in report.correlations:
    status = "PASS" if c.passed else "FAIL"
    print(
        f"{c.field1} ~ {c.field2}: target={c.target:+.2f} "
        f"observed(spearman)={c.observed_spearman:+.2f}  [{status}]"
    )

## Visualising the fit

The KS test compares the **empirical CDF** of the samples to the distribution's **theoretical CDF** — its statistic is the largest gap between the two curves. For a categorical field, the chi-square test compares observed and expected frequencies. Both are easy to see directly.

In [ ]:
specs = LLMDrivenModelAnalyser.extract_distribution_specs(Employee)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Continuous: empirical vs theoretical CDF (exactly what the KS test compares)
salary = np.sort([r["salary"] for r in records])
ecdf = np.arange(1, len(salary) + 1) / len(salary)
axes[0].plot(salary, ecdf, label="empirical", lw=2)
axes[0].plot(salary, specs["salary"].cdf(salary), label="theoretical", lw=2, ls="--")
axes[0].set_title("salary: empirical vs theoretical CDF")
axes[0].set_xlabel("salary")
axes[0].set_ylabel("cumulative probability")
axes[0].legend()

# Categorical: observed vs expected frequencies
weights = specs["department"].weights
cats = list(weights.keys())
observed = [sum(r["department"] == c for r in records) / len(records) for c in cats]
expected = [weights[c] for c in cats]
x = np.arange(len(cats))
axes[1].bar(x - 0.2, observed, width=0.4, label="observed")
axes[1].bar(x + 0.2, expected, width=0.4, label="expected")
axes[1].set_xticks(x)
axes[1].set_xticklabels(cats)
axes[1].set_title("department: observed vs expected frequency")
axes[1].legend()

plt.tight_layout()
plt.show()

## Catching a regression: a shifted marginal

Imagine a sampling bug halves every salary. The marginal no longer matches `Normal(75000, 20000)`, so the KS test on `salary` fails. Note the `age ~ salary` correlation **still passes** — halving is monotonic, so the rank correlation is untouched. Fidelity checks the *shape* and the *relationship* independently.

In [ ]:
# Simulate a sampling bug: salaries drawn far too low.
broken = [dict(r) for r in records]
for r in broken:
    r["salary"] = r["salary"] * 0.5

broken_report = fidelity_report(broken, Employee)
print(broken_report)

## Catching a regression: a broken relationship

Now the opposite failure — the marginals are all correct, but the `age`/`salary` *relationship* is gone. Shuffling `salary` keeps its distribution identical (every field still passes) while destroying the correlation, so only the correlation check fails.

In [ ]:
# Shuffle salary: same marginal distribution, but the age~salary link is destroyed.
shuffled = [dict(r) for r in records]
salaries = np.array([r["salary"] for r in shuffled])
np.random.default_rng(0).shuffle(salaries)
for r, s in zip(shuffled, salaries):
    r["salary"] = float(s)

shuffled_report = fidelity_report(shuffled, Employee)
print(shuffled_report)

## Tuning the verdict

Two knobs control how strict the report is:

- **`alpha`** (default `0.05`) — the significance level for the goodness-of-fit tests. A field passes when its p-value is `>= alpha`. A *larger* `alpha` is *stricter* (rejects more readily).
- **`correlation_tolerance`** (default `0.15`) — the maximum allowed gap between the observed Spearman correlation and the declared target.

In [ ]:
for alpha in (0.01, 0.05, 0.20):
    rep = fidelity_report(records, Employee, alpha=alpha)
    print(f"alpha={alpha:<4}  passed={rep.passed}")

## When to use it

- **In tests / CI** — assert `fidelity_report(records, Model).passed` to catch regressions in the sampling or copula machinery.
- **When tuning a model** — eyeball the summary to confirm your distributions and correlations produce what you intended.
- **As a data-quality gate** — validate a batch before handing it downstream.

Only distribution-annotated fields and declared `__correlations__` are checked; free-text (LLM-generated) fields are ignored, since there's no spec to compare them against.